# Case 3: Caspase Active & Viability Inactive

Analysis of compounds where:
- **Caspase = Active** (has at least one Active outcome)
- **Viability = Inactive** (has at least one Inactive outcome AND no Active outcomes)

**Classification Method:** STRICT
- Active takes precedence: if ANY test shows Active, the CID is classified as Active
- Inactive requires explicit Inactive outcome (not just absence of Active)
- CIDs with only Inconclusive outcomes are excluded

**Biological Interpretation:** Pure apoptosis (selective apoptosis inducers)

**Assay Types:**
- `caspase` - Caspase-3/7 assay (14 cell lines)
- `viability` - Viability assay (CHO-K1)
- `mitochondrial_viability_hepg2` - Mitochondrial viability (HepG2)
- `caspase_viability_hepg2` - Caspase viability counter-screen (HepG2)

In [21]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.max_rows', 100)

# Load the merged dataset
df = pd.read_csv('output/merged_caspase_viability_final.csv')

print(f"Dataset Shape: {df.shape}")
print(f"Unique CIDs: {df['CID'].nunique():,}")
print(f"\nAssay Types: {df['assay_type'].unique().tolist()}")
print(f"\nCell Lines ({df['cell_line'].nunique()}): {df['cell_line'].unique().tolist()}")

Dataset Shape: (26939, 12)
Unique CIDs: 8,777

Assay Types: ['viability', 'caspase', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

Cell Lines (14): ['CHO-K1', 'HepG2', 'SHSY5Y', 'Hek293', 'HUV-EC-C', 'Jurkat', 'H-4-II-E', 'N2a', 'Mesangial', 'NIH 3T3', 'SK-N-SH', 'BJ', 'MRC-5', 'Renal_Proximal_Tubule']


In [22]:
# Define viability types
VIABILITY_TYPES = ['viability', 'mitochondrial_viability_hepg2', 'caspase_viability_hepg2']

print("Viability assay types used in this analysis:")
print(f"  - viability (CHO-K1)")
print(f"  - mitochondrial_viability_hepg2 (HepG2)")
print(f"  - caspase_viability_hepg2 (HepG2)")

Viability assay types used in this analysis:
  - viability (CHO-K1)
  - mitochondrial_viability_hepg2 (HepG2)
  - caspase_viability_hepg2 (HepG2)


---
## Block 1: Overview - Case 3 Criteria (STRICT Method)

Find CIDs where:
- Caspase = **Active** (has at least one Active outcome)
- Viability = **Inactive** (has at least one Inactive outcome AND no Active outcomes)

**Note:** Only CIDs tested in BOTH assay types are included.

In [23]:
# STRICT METHOD: 
# - Active takes precedence
# - Inactive = has Inactive outcome AND no Active outcome
# Case 3 = Caspase Active AND Viability Truly Inactive

# Get all CIDs tested in caspase assays
caspase_df = df[df['assay_type'] == 'caspase']
caspase_all_cids = set(caspase_df['CID'].unique())
caspase_active_cids = set(caspase_df[caspase_df['outcome'] == 'Active']['CID'].unique())

# Get all CIDs tested in viability assays
viability_df = df[df['assay_type'].isin(VIABILITY_TYPES)]
viability_all_cids = set(viability_df['CID'].unique())
viability_active_cids = set(viability_df[viability_df['outcome'] == 'Active']['CID'].unique())
viability_inactive_cids = set(viability_df[viability_df['outcome'] == 'Inactive']['CID'].unique())

# Common CIDs - tested in BOTH assay types
common_cids = caspase_all_cids.intersection(viability_all_cids)

# Caspase Active in common CIDs
caspase_active_in_common = caspase_active_cids.intersection(common_cids)

# Viability Truly Inactive = has Inactive outcome but NO Active outcome (in common CIDs)
viability_truly_inactive = viability_inactive_cids.intersection(common_cids) - viability_active_cids

print("STRICT METHOD: Active takes precedence, Inactive = has Inactive & no Active")
print("="*70)
print(f"CIDs tested in Caspase assays:     {len(caspase_all_cids):,}")
print(f"CIDs tested in Viability assays:   {len(viability_all_cids):,}")
print(f"CIDs tested in BOTH (common):      {len(common_cids):,}")
print()
print(f"Caspase Active CIDs (in common):   {len(caspase_active_in_common):,}")
print(f"Viability Active CIDs:             {len(viability_active_cids):,}")
print(f"Viability Inactive CIDs:           {len(viability_inactive_cids):,}")
print(f"Viability Truly Inactive (in common): {len(viability_truly_inactive):,}")
print()

# Case 3: Caspase Active AND Viability Truly Inactive
case3_cids = caspase_active_in_common.intersection(viability_truly_inactive)

print(f"CASE 3: Caspase=Active AND Viability=Inactive: {len(case3_cids):,}")
print()
print("These compounds induce PURE APOPTOSIS (selective apoptosis inducers)")

STRICT METHOD: Active takes precedence, Inactive = has Inactive & no Active
CIDs tested in Caspase assays:     7,716
CIDs tested in Viability assays:   8,734
CIDs tested in BOTH (common):      7,673

Caspase Active CIDs (in common):   318
Viability Active CIDs:             1,273
Viability Inactive CIDs:           8,213
Viability Truly Inactive (in common): 6,367

CASE 3: Caspase=Active AND Viability=Inactive: 53

These compounds induce PURE APOPTOSIS (selective apoptosis inducers)


In [24]:
# Show these CIDs
print(f"List of {len(case3_cids):,} CIDs with Caspase=Active AND Viability=Never Active:")
print(sorted(case3_cids)[:100])  # Show first 100
if len(case3_cids) > 100:
    print(f"... and {len(case3_cids) - 100} more")

List of 53 CIDs with Caspase=Active AND Viability=Never Active:
[311, 1674, 2082, 2272, 3485, 4122, 4211, 4993, 5054, 5144, 6108, 6294, 6478, 6720, 7017, 8640, 9158, 11852, 11990, 12232, 12302, 13636, 16666, 18048, 18177, 21013, 25429, 36462, 60699, 60822, 61179, 62630, 62753, 88365, 119369, 197873, 441140, 443101, 446155, 1269845, 2723598, 2724189, 2724385, 2725002, 5282164, 5282451, 5284441, 6063342, 6440940, 9849425, 11159021, 21935081, 23663992]


---
## Block 2: Case 3 Analysis by Cell Line (SAME CELL LINE)

For each cell line, find CIDs where Caspase=Active AND Viability=Inactive in the SAME cell line.

In [25]:
# Analysis by Cell Line - Caspase Active & Viability Inactive (SAME CELL LINE)
print("="*70)
print("CASE 3: CASPASE=ACTIVE AND VIABILITY=INACTIVE BY CELL LINE")
print("="*70)
print("Note: Viability data exists only for HepG2 and CHO-K1")
print()

# Get caspase and viability dataframes
caspase_df = df[df['assay_type'] == 'caspase']
viability_df = df[df['assay_type'].isin(VIABILITY_TYPES)]

caspase_cell_lines = sorted(caspase_df['cell_line'].unique())

results = []
for cell in caspase_cell_lines:
    # Total CIDs tested for caspase in this cell line
    caspase_in_cell = caspase_df[caspase_df['cell_line'] == cell]
    total_caspase_tested = caspase_in_cell['CID'].nunique()
    
    # Caspase Active in this cell line
    casp_active = set(caspase_in_cell[caspase_in_cell['outcome'] == 'Active']['CID'].unique())
    
    # Viability Inactive in THIS SAME cell line
    viability_in_cell = viability_df[viability_df['cell_line'] == cell]
    total_viability_tested = viability_in_cell['CID'].nunique()
    viab_inactive = set(viability_in_cell[viability_in_cell['outcome'] == 'Inactive']['CID'].unique())
    
    # Case 3: Caspase Active AND Viability Inactive in SAME cell line
    case3_in_cell = casp_active.intersection(viab_inactive)
    
    results.append({
        'Cell Line': cell,
        'Caspase Tested': total_caspase_tested,
        'Caspase Active': len(casp_active),
        'Viability Tested': total_viability_tested,
        'Viability Inactive': len(viab_inactive),
        'Case3 (Both)': len(case3_in_cell),
        'Case3 Rate (%)': round(len(case3_in_cell)/len(casp_active)*100, 2) if len(casp_active) > 0 else 0
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('Caspase Active', ascending=False)
print(results_df.to_string(index=False))

# Summary for cell lines WITH viability data
print(f"\n" + "-"*70)
hepg2_case3 = results_df[results_df['Cell Line'] == 'HepG2']['Case3 (Both)'].values[0]
chok1_case3 = results_df[results_df['Cell Line'] == 'CHO-K1']['Case3 (Both)'].values[0]
print(f"HepG2: {hepg2_case3} CIDs with Caspase=Active AND Viability=Inactive")
print(f"CHO-K1: {chok1_case3} CIDs with Caspase=Active AND Viability=Inactive")

CASE 3: CASPASE=ACTIVE AND VIABILITY=INACTIVE BY CELL LINE
Note: Viability data exists only for HepG2 and CHO-K1

            Cell Line  Caspase Tested  Caspase Active  Viability Tested  Viability Inactive  Case3 (Both)  Case3 Rate (%)
                HepG2            7671             191              8734                8200           115           60.21
               CHO-K1             394              61              7671                5743             4            6.56
               Jurkat             106              32                 0                   0             0            0.00
             H-4-II-E              24              11                 0                   0             0            0.00
               Hek293              19               8                 0                   0             0            0.00
              SK-N-SH              17               7                 0                   0             0            0.00
              NIH 3T3           

In [26]:
# HepG2 Specific: Case 3 IN THE SAME CELL LINE
print("="*70)
print("HepG2: CASPASE=ACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)")
print("="*70)

# Caspase Active in HepG2
caspase_active_hepg2 = set(df[
    (df['assay_type'] == 'caspase') & 
    (df['cell_line'] == 'HepG2') & 
    (df['outcome'] == 'Active')
]['CID'].unique())

# Viability Inactive in HepG2 (mitochondrial_viability_hepg2 + caspase_viability_hepg2)
viability_inactive_hepg2 = set(df[
    (df['assay_type'].isin(['mitochondrial_viability_hepg2', 'caspase_viability_hepg2'])) & 
    (df['cell_line'] == 'HepG2') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

case3_hepg2 = caspase_active_hepg2.intersection(viability_inactive_hepg2)

print(f"\nCaspase Active in HepG2: {len(caspase_active_hepg2):,}")
print(f"Viability Inactive in HepG2 (mito + caspase_viability): {len(viability_inactive_hepg2):,}")
print(f"\n>>> CASE 3 in HepG2: {len(case3_hepg2):,}")

if len(case3_hepg2) > 0:
    print(f"\nCIDs: {sorted(case3_hepg2)}")

HepG2: CASPASE=ACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)

Caspase Active in HepG2: 191
Viability Inactive in HepG2 (mito + caspase_viability): 8,200

>>> CASE 3 in HepG2: 115

CIDs: [1923, 2082, 2662, 3109, 3117, 3191, 3559, 3634, 4122, 4257, 4807, 5054, 5144, 5311, 5405, 5455, 5746, 5991, 6498, 6529, 6536, 6928, 7263, 7347, 7352, 7418, 7948, 8117, 8425, 8441, 8455, 9158, 9403, 10788, 10866, 11065, 11742, 11990, 12170, 12232, 14642, 16666, 17004, 17170, 18048, 18980, 19658, 25429, 26041, 26633, 26710, 28780, 30323, 31426, 33528, 33630, 36462, 51082, 53232, 60699, 60822, 61080, 62556, 62857, 63009, 66030, 68539, 68570, 71478, 71612, 82146, 119369, 154914, 171144, 208908, 387447, 441140, 443101, 446155, 456201, 517546, 560256, 636362, 1269845, 2723853, 2725002, 3005572, 5282164, 5282451, 6063342, 6434889, 6456015, 6708701, 9849425, 9909677, 9954540, 11159021, 11296583, 11477651, 11556711, 11626384, 12942137, 14552708, 21935081, 23663992, 44235958, 46780884, 60196411, 60196435, 76973

In [27]:
# CHO-K1 Specific: Case 3 IN THE SAME CELL LINE
print("="*70)
print("CHO-K1: CASPASE=ACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)")
print("="*70)

# Caspase Active in CHO-K1
caspase_active_chok1 = set(df[
    (df['assay_type'] == 'caspase') & 
    (df['cell_line'] == 'CHO-K1') & 
    (df['outcome'] == 'Active')
]['CID'].unique())

# Viability Inactive in CHO-K1
viability_inactive_chok1 = set(df[
    (df['assay_type'] == 'viability') & 
    (df['cell_line'] == 'CHO-K1') & 
    (df['outcome'] == 'Inactive')
]['CID'].unique())

case3_chok1 = caspase_active_chok1.intersection(viability_inactive_chok1)

print(f"\nCaspase Active in CHO-K1: {len(caspase_active_chok1):,}")
print(f"Viability Inactive in CHO-K1: {len(viability_inactive_chok1):,}")
print(f"\n>>> CASE 3 in CHO-K1: {len(case3_chok1):,}")

if len(case3_chok1) > 0:
    print(f"\nCIDs: {sorted(case3_chok1)}")

CHO-K1: CASPASE=ACTIVE AND VIABILITY=INACTIVE (SAME CELL LINE)

Caspase Active in CHO-K1: 61
Viability Inactive in CHO-K1: 5,743

>>> CASE 3 in CHO-K1: 4

CIDs: [8461, 15659, 31362, 62630]


---
## Block 3: Export Data - Case 3

Save the data to CSV for easy verification, grouped by CID.

In [28]:
# Case 3: Export CIDs where Caspase=Active AND Viability=Inactive (truly)
print("="*70)
print("CASE 3: EXPORTING CASPASE ACTIVE & VIABILITY INACTIVE DATA")
print("="*70)

print(f"\nTotal CIDs (strict method): {len(case3_cids):,}")

# Get records for these CIDs:
# - Caspase Active records
# - All Viability records (to show outcomes - should have Inactive but no Active)
case3_caspase = df[
    (df['CID'].isin(case3_cids)) & 
    (df['assay_type'] == 'caspase') &
    (df['outcome'] == 'Active')
][['CID', 'SMILES', 'assay_type', 'cell_line', 'outcome', 'activity_score', 'AID', 'assay_name']].copy()

case3_viability = df[
    (df['CID'].isin(case3_cids)) & 
    (df['assay_type'].isin(VIABILITY_TYPES))
][['CID', 'SMILES', 'assay_type', 'cell_line', 'outcome', 'activity_score', 'AID', 'assay_name']].copy()

# Combine
case3_data = pd.concat([case3_caspase, case3_viability], ignore_index=True)
case3_data = case3_data.sort_values(['CID', 'cell_line', 'assay_type'])

print(f"Total records: {len(case3_data):,}")
print(f"\nPreview:")
print(case3_data.head(20).to_string(index=False))

# Verify: Viability outcomes should have Inactive but NO Active
viability_outcomes = case3_viability['outcome'].value_counts()
print(f"\nViability outcomes for Case 3 CIDs (should have Inactive, NO Active):")
print(viability_outcomes)

# Save to CSV
output_path = 'output/case3_caspase_active_viability_inactive.csv'
case3_data.to_csv(output_path, index=False)
print(f"\n>>> Saved to: {output_path}")
print(f"    Shape: {case3_data.shape}")

CASE 3: EXPORTING CASPASE ACTIVE & VIABILITY INACTIVE DATA

Total CIDs (strict method): 53
Total records: 187

Preview:
 CID                                      SMILES                    assay_type cell_line      outcome  activity_score     AID                   assay_name
 311                C(C(=O)O)C(CC(=O)O)(C(=O)O)O                     viability    CHO-K1     Inactive               0 1346979 AID_1346979_viability_CHO-K1
 311                C(C(=O)O)C(CC(=O)O)(C(=O)O)O mitochondrial_viability_hepg2     HepG2     Inactive               0  720634        AID_720634_mito_HepG2
 311                C(C(=O)O)C(CC(=O)O)(C(=O)O)O                       caspase    Jurkat       Active              73     655      AID_655_caspase3_Jurkat
1674  CC1=C2CCC3=C2C(=CC4=C3C=CC5=CC=CC=C54)C=C1                     viability    CHO-K1     Inactive               0 1346979 AID_1346979_viability_CHO-K1
1674  CC1=C2CCC3=C2C(=CC4=C3C=CC5=CC=CC=C54)C=C1                       caspase     HepG2       Active    

In [29]:
# Create summary table grouped by CID for easy count verification
print("="*70)
print("CASE 3: SUMMARY BY CID (STRICT METHOD)")
print("="*70)

# For each CID, get the caspase active cell lines and viability outcomes
caspase_active_by_cid = case3_caspase.groupby('CID').agg({
    'cell_line': lambda x: ', '.join(sorted(x.unique())),
    'activity_score': 'max'
}).reset_index()
caspase_active_by_cid.columns = ['CID', 'Caspase_Active_CellLines', 'Max_Caspase_Score']

viability_by_cid = case3_viability.groupby('CID').agg({
    'cell_line': lambda x: ', '.join(sorted(x.unique())),
    'assay_type': lambda x: ', '.join(sorted(x.unique())),
    'outcome': lambda x: ', '.join(sorted(x.unique()))
}).reset_index()
viability_by_cid.columns = ['CID', 'Viability_CellLines', 'Viability_AssayTypes', 'Viability_Outcomes']

# Merge
case3_summary = pd.merge(caspase_active_by_cid, viability_by_cid, on='CID')

# Add SMILES
smiles_map = df[['CID', 'SMILES']].drop_duplicates().set_index('CID')['SMILES'].to_dict()
case3_summary['SMILES'] = case3_summary['CID'].map(smiles_map)

case3_summary = case3_summary.sort_values('CID')

print(f"\nTotal unique CIDs: {len(case3_summary):,}")
print(f"\nSummary table (first 30 rows):")
print(case3_summary.head(30).to_string(index=False))

# Save summary to CSV
summary_path = 'output/case3_summary_by_cid.csv'
case3_summary.to_csv(summary_path, index=False)
print(f"\n>>> Summary saved to: {summary_path}")
print(f"    Total CIDs: {len(case3_summary):,}")

CASE 3: SUMMARY BY CID (STRICT METHOD)

Total unique CIDs: 53

Summary table (first 30 rows):
  CID Caspase_Active_CellLines  Max_Caspase_Score Viability_CellLines                                              Viability_AssayTypes     Viability_Outcomes                                                                                                                                                                                         SMILES
  311                   Jurkat                 73       CHO-K1, HepG2                          mitochondrial_viability_hepg2, viability               Inactive                                                                                                                                                                   C(C(=O)O)C(CC(=O)O)(C(=O)O)O
 1674                    HepG2                 85       CHO-K1, HepG2                                caspase_viability_hepg2, viability Inactive, Inconclusive                                                